# Advanced Module 2 · Agentic Loops & ReAct
Agents that reason in loops — from a bare ReAct loop to self-correcting multi-step agents.

---
> **Prerequisite:** Complete Module 1 (Function Calling & Tools) first.  
> **Setup:** Run the first two cells before anything else.

In [16]:
%pip install -q google-genai groq python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [17]:
import os, json, math, time, base64, getpass, re
from dotenv import load_dotenv
from google import genai
from google.genai import types
from groq import Groq

load_dotenv()
gemini_api_key = os.environ.get('GEMINI_API_KEY')
if not gemini_api_key:
    gemini_api_key = getpass.getpass('Paste your Gemini API key: ')

groq_api_key = os.environ.get('Groq_key', '').strip()
if not groq_api_key:
    groq_api_key = getpass.getpass('Paste your Groq API key: ')

gemini_client = genai.Client(api_key=gemini_api_key)
groq_client   = Groq(api_key=groq_api_key)
GEMINI_MODEL  = 'gemini-3.1-flash-lite'
GROQ_MODEL    = 'llama-3.3-70b-versatile'
DEFAULT_MAX_OUTPUT_TOKENS = 2048

def make_generation_config(**kwargs):
    """Build a GenerateContentConfig with sensible defaults.

    temperature=0.0  → deterministic; best for tool calls and routing decisions.
    temperature=0.7  → creative; best for final natural-language answers.
    """
    kwargs.setdefault('max_output_tokens', DEFAULT_MAX_OUTPUT_TOKENS)
    kwargs.setdefault('temperature', 0.7)
    return types.GenerateContentConfig(**kwargs)

def extract_text_from_response(response) -> str:
    """Pull the final answer text from a Gemini response, skipping reasoning thoughts.

    Reasoning models emit internal 'thought' parts before the final answer.
    This function skips those and returns only the user-facing text.
    """
    if response.text:
        return response.text.strip()
    text_parts = []
    for candidate in (response.candidates or []):
        if candidate.content:
            for part in candidate.content.parts:
                if not getattr(part, 'thought', False) and part.text:
                    text_parts.append(part.text)
    return ''.join(text_parts).strip()

def call_with_retry(api_function, *args, max_retries=5, **kwargs):
    """Wrap an API call with automatic retry for rate-limit and server errors."""
    for attempt in range(max_retries):
        try:
            return api_function(*args, **kwargs)
        except Exception as error:
            error_message = str(error)
            if '429' in error_message or 'RESOURCE_EXHAUSTED' in error_message:
                retry_wait_match = re.search(r'retry[^0-9]*([0-9]+)s', error_message, re.I)
                wait_seconds = int(retry_wait_match.group(1)) + 5 if retry_wait_match else 35
                print(f'  ⏳ Rate-limited — waiting {wait_seconds}s (attempt {attempt+1}/{max_retries})')
                time.sleep(wait_seconds)
            elif '500' in error_message or 'INTERNAL' in error_message:
                wait_seconds = 10 * (attempt + 1)
                print(f'  ⏳ Server error — waiting {wait_seconds}s (attempt {attempt+1}/{max_retries})')
                time.sleep(wait_seconds)
            else:
                raise
    raise RuntimeError('Max retries exceeded')

original_generate_content = gemini_client.models.generate_content
gemini_client.models.generate_content = lambda *args, **kwargs: call_with_retry(original_generate_content, *args, **kwargs)

print('✅ Setup complete. Gemini:', GEMINI_MODEL, '| Groq:', GROQ_MODEL)

✅ Setup complete. Gemini: gemini-3.1-flash-lite | Groq: llama-3.3-70b-versatile


In [18]:
try:
    test_response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents='Reply with exactly the words: Hello LLM course',
        config=make_generation_config(temperature=0.0)
    )
    print('✅ Gemini API working:', extract_text_from_response(test_response))
except Exception as error:
    print('❌ Gemini error:', error)

✅ Gemini API working: Hello LLM course


---
## The Core Idea: An Agent Is Just a While-Loop

In Module 1 we saw single and chained tool calls. An **agent** takes this further:  
it loops — calling tools, observing results, reasoning about them — until the task is done.

**ReAct** (Reason + Act) is the most popular pattern: the model alternates between  
writing a **Thought** (reasoning) and taking an **Action** (tool call), then observing the result.

```
┌─────────────────────────────────────────────────────────────────┐
│                     THE REACT AGENT LOOP                        │
│                                                                 │
│   User task: "Book cheapest flight + 3-night hotel"             │
│          │                                                      │
│          ▼                                                      │
│   ┌─────────────┐                                               │
│   │  THOUGHT 1  │  "I need to search for flights first"         │
│   └──────┬──────┘                                               │
│          ▼                                                      │
│   ┌─────────────┐                                               │
│   │  ACTION 1   │  search_flights(origin='BOM', dest='BER')     │
│   └──────┬──────┘                                               │
│          ▼                                                      │
│   ┌─────────────┐                                               │
│   │ OBSERVATION │  [{'id':'F1','price':45000}, ...]             │
│   └──────┬──────┘                                               │
│          ▼                                                      │
│   ┌─────────────┐                                               │
│   │  THOUGHT 2  │  "F1 is cheapest. Now I need a hotel"         │
│   └──────┬──────┘                                               │
│          ▼                                                      │
│   ┌─────────────┐                                               │
│   │  ACTION 2   │  search_hotels(city='Berlin', nights=3)       │
│   └──────┬──────┘                                               │
│          ▼                                                      │
│   ┌─────────────┐                                               │
│   │ OBSERVATION │  [{'id':'H2','price_per_night':8500}, ...]    │
│   └──────┬──────┘                                               │
│          ▼                                                      │
│   ┌─────────────┐                                               │
│   │ FINAL ANSWER│  "Cheapest option: F1 + H2 = ₹70,500 total"  │
│   └─────────────┘                                               │
└─────────────────────────────────────────────────────────────────┘
```

Key difference from Module 1: **the agent decides dynamically** which tools to call and in what order.  
No hardcoded sequence — just reasoning.

---
## Demo 1: Bare ReAct Loop from Scratch

No frameworks. A pure Python while-loop. Every Thought and Action printed explicitly.

In [19]:
def get_weather(city: str) -> dict:
    """Return stub weather data for a city."""
    weather_database = {
        'mumbai':  {'temp': 32, 'condition': 'humid',  'rain_chance': 80},
        'berlin':  {'temp': 18, 'condition': 'cloudy', 'rain_chance': 30},
        'london':  {'temp': 15, 'condition': 'rainy',  'rain_chance': 75},
        'dubai':   {'temp': 42, 'condition': 'sunny',  'rain_chance': 2},
        'toronto': {'temp': 22, 'condition': 'clear',  'rain_chance': 10},
    }
    return weather_database.get(city.lower(), {'temp': 20, 'condition': 'unknown', 'rain_chance': 50})

def convert_currency(amount: float, from_currency: str, to_currency: str) -> dict:
    """Convert an amount between currencies using fixed stub exchange rates."""
    exchange_rates = {'USD': 1.0, 'EUR': 0.93, 'GBP': 0.79, 'INR': 83.5, 'AED': 3.67}
    amount_in_usd  = amount / exchange_rates.get(from_currency.upper(), 1.0)
    converted      = round(amount_in_usd * exchange_rates.get(to_currency.upper(), 1.0), 2)
    return {'from': f'{amount} {from_currency}', 'to': f'{converted} {to_currency}'}

def get_flight_info(origin: str, destination: str) -> dict:
    """Return stub flight information between two cities."""
    return {
        'route':             f'{origin} → {destination}',
        'duration_hours':    9,
        'cheapest_fare_usd': 620,
        'airline':           'SkyWings',
    }

# Map each tool name to its Python callable.
# The agent loop uses this dict to dispatch: TRAVEL_TOOL_MAP[function_call.name](**args)
TRAVEL_TOOL_MAP = {
    'get_weather':      get_weather,
    'convert_currency': convert_currency,
    'get_flight_info':  get_flight_info,
}

travel_tool_schema = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name='get_weather',
            description='Get current weather for a city.',
            parameters=types.Schema(type=types.Type.OBJECT,
                properties={'city': types.Schema(type=types.Type.STRING)},
                required=['city'])
        ),
        types.FunctionDeclaration(
            name='convert_currency',
            description='Convert an amount from one currency to another.',
            parameters=types.Schema(type=types.Type.OBJECT,
                properties={
                    'amount':        types.Schema(type=types.Type.NUMBER),
                    'from_currency': types.Schema(type=types.Type.STRING, description='e.g. USD, EUR, INR'),
                    'to_currency':   types.Schema(type=types.Type.STRING, description='e.g. USD, EUR, INR'),
                },
                required=['amount', 'from_currency', 'to_currency'])
        ),
        types.FunctionDeclaration(
            name='get_flight_info',
            description='Get flight information between two cities.',
            parameters=types.Schema(type=types.Type.OBJECT,
                properties={
                    'origin':      types.Schema(type=types.Type.STRING),
                    'destination': types.Schema(type=types.Type.STRING),
                },
                required=['origin', 'destination'])
        ),
    ]
)

print('✅ Tools ready:', list(TRAVEL_TOOL_MAP.keys()))

✅ Tools ready: ['get_weather', 'convert_currency', 'get_flight_info']


In [20]:
def run_react_agent(task: str, max_steps: int = 8, verbose: bool = True) -> str:
    """Run a ReAct (Reason + Act) agent loop using the travel tools.

    The loop works like this:
      1. Send the conversation history (including all past tool results) to the LLM.
      2. If the LLM returns one or more function_call parts → execute them, append
         results to history, go to step 1.
      3. If the LLM returns text with no function_call → that is the final answer. Stop.

    The agent decides at each step which tool(s) to call and in what order.
    We never hardcode a sequence — the model reasons dynamically.

    max_steps guards against runaway loops that would burn tokens indefinitely.
    """
    conversation  = [types.Content(role='user', parts=[types.Part(text=task)])]
    total_tokens  = 0
    step_count    = 0

    if verbose:
        print('=' * 65)
        print(f'TASK: {task}')
        print('=' * 65)

    while step_count < max_steps:
        step_count += 1
        llm_response = gemini_client.models.generate_content(
            model=GEMINI_MODEL,
            contents=conversation,
            config=make_generation_config(tools=[travel_tool_schema], temperature=0.0)
        )
        total_tokens += llm_response.usage_metadata.total_token_count

        # Collect every function_call the model requested in this turn.
        # A single response can contain multiple parallel tool requests.
        all_function_calls = [
            part.function_call
            for part in llm_response.candidates[0].content.parts
            if hasattr(part, 'function_call') and part.function_call
        ]

        if not all_function_calls:
            # No tool calls means the model has enough information to answer.
            final_answer = extract_text_from_response(llm_response)
            if verbose:
                print(f'\n[Step {step_count}] THOUGHT → Final answer (no more tools needed)')
                print(f'[Step {step_count}] ANSWER  → {final_answer}')
                print(f'\n📊 Total steps: {step_count} | Total tokens used: {total_tokens}')
            return final_answer

        # Execute every requested tool and collect results
        tool_results = []
        for function_call in all_function_calls:
            result = TRAVEL_TOOL_MAP[function_call.name](**dict(function_call.args))
            tool_results.append(result)
            if verbose:
                print(f'[Step {step_count}] ACTION      → {function_call.name}({dict(function_call.args)})')
                print(f'[Step {step_count}] OBSERVATION → {result}')

        # Append the model turn first (preserves thought_signature), then all tool results
        conversation.append(llm_response.candidates[0].content)
        conversation.append(types.Content(role='user', parts=[
            types.Part(function_response=types.FunctionResponse(
                name=function_call.name,
                response={'result': result}
            ))
            for function_call, result in zip(all_function_calls, tool_results)
        ]))

    return 'Max steps reached without a final answer.'

result = run_react_agent(
    "I'm in Mumbai and want to fly to London. What's the weather like there, "
    "and what will the flight cost in Indian Rupees?"
)

TASK: I'm in Mumbai and want to fly to London. What's the weather like there, and what will the flight cost in Indian Rupees?
[Step 1] ACTION      → get_weather({'city': 'London'})
[Step 1] OBSERVATION → {'temp': 15, 'condition': 'rainy', 'rain_chance': 75}
[Step 1] ACTION      → get_flight_info({'destination': 'London', 'origin': 'Mumbai'})
[Step 1] OBSERVATION → {'route': 'Mumbai → London', 'duration_hours': 9, 'cheapest_fare_usd': 620, 'airline': 'SkyWings'}
[Step 2] ACTION      → convert_currency({'from_currency': 'USD', 'amount': 620, 'to_currency': 'INR'})
[Step 2] OBSERVATION → {'from': '620 USD', 'to': '51770.0 INR'}

[Step 3] THOUGHT → Final answer (no more tools needed)
[Step 3] ANSWER  → The weather in London is currently rainy with a temperature of 15°C and a 75% chance of rain.

Regarding your travel, a flight from Mumbai to London with SkyWings takes approximately 9 hours. The cheapest fare is 620 USD, which is approximately 51,770 INR.

📊 Total steps: 3 | Total tokens us

---
## Demo 2: Multi-Step Agent — Trip Cost Calculator

A realistic multi-step task: find the cheapest flight, look up hotel prices,  
and calculate total trip cost in the user's currency.

In [21]:
import random

def search_flights(origin: str, destination: str, date: str) -> dict:
    """Return stub flight options for a route and date."""
    return {
        'route': f'{origin}→{destination}',
        'date':  date,
        'options': [
            {'flight_id': 'SK201', 'airline': 'SkyWings', 'price_usd': 580, 'duration_h': 9},
            {'flight_id': 'AE445', 'airline': 'AeroEast', 'price_usd': 640, 'duration_h': 11},
            {'flight_id': 'SW900', 'airline': 'SwiftAir', 'price_usd': 520, 'duration_h': 10},
        ]
    }

def search_hotels(city: str, checkin: str, checkout: str) -> dict:
    """Return stub hotel options for a city and date range."""
    return {
        'city': city, 'checkin': checkin, 'checkout': checkout,
        'options': [
            {'hotel_id': 'H01', 'name': 'CityInn',    'price_per_night_usd': 85,  'rating': 3.8},
            {'hotel_id': 'H02', 'name': 'ParkView',   'price_per_night_usd': 140, 'rating': 4.3},
            {'hotel_id': 'H03', 'name': 'BudgetStay', 'price_per_night_usd': 55,  'rating': 3.2},
        ]
    }

def calculate_total_trip_cost(
    flight_price_usd: float,
    hotel_price_per_night_usd: float,
    nights: int
) -> dict:
    """Calculate total trip cost in USD and INR given flight and hotel prices."""
    hotel_total_usd = hotel_price_per_night_usd * nights
    grand_total_usd = flight_price_usd + hotel_total_usd
    inr_per_usd     = 83.5
    return {
        'flight_usd':  flight_price_usd,
        'hotel_usd':   hotel_total_usd,
        'total_usd':   grand_total_usd,
        'total_inr':   round(grand_total_usd * inr_per_usd, 0),
        'nights':      nights,
    }

TRIP_TOOL_MAP = {
    'search_flights':           search_flights,
    'search_hotels':            search_hotels,
    'calculate_total_trip_cost': calculate_total_trip_cost,
}

trip_planning_tool_schema = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name='search_flights',
            description='Search available flights between two cities on a given date.',
            parameters=types.Schema(type=types.Type.OBJECT,
                properties={
                    'origin':      types.Schema(type=types.Type.STRING),
                    'destination': types.Schema(type=types.Type.STRING),
                    'date':        types.Schema(type=types.Type.STRING, description='YYYY-MM-DD'),
                },
                required=['origin', 'destination', 'date'])
        ),
        types.FunctionDeclaration(
            name='search_hotels',
            description='Search available hotels in a city for given check-in and check-out dates.',
            parameters=types.Schema(type=types.Type.OBJECT,
                properties={
                    'city':     types.Schema(type=types.Type.STRING),
                    'checkin':  types.Schema(type=types.Type.STRING, description='YYYY-MM-DD'),
                    'checkout': types.Schema(type=types.Type.STRING, description='YYYY-MM-DD'),
                },
                required=['city', 'checkin', 'checkout'])
        ),
        types.FunctionDeclaration(
            name='calculate_total_trip_cost',
            description='Calculate total trip cost given flight price, hotel nightly rate, and number of nights.',
            parameters=types.Schema(type=types.Type.OBJECT,
                properties={
                    'flight_price_usd':           types.Schema(type=types.Type.NUMBER),
                    'hotel_price_per_night_usd':  types.Schema(type=types.Type.NUMBER),
                    'nights':                     types.Schema(type=types.Type.INTEGER),
                },
                required=['flight_price_usd', 'hotel_price_per_night_usd', 'nights'])
        ),
    ]
)

def run_trip_planning_agent(task: str, max_steps: int = 10) -> str:
    """Agent that plans a full trip: searches flights, hotels, then calculates cost."""
    conversation = [types.Content(role='user', parts=[types.Part(text=task)])]
    total_tokens = 0
    print('=' * 65)
    print(f'TASK: {task}')
    print('=' * 65)

    for step_number in range(1, max_steps + 1):
        llm_response = gemini_client.models.generate_content(
            model=GEMINI_MODEL,
            contents=conversation,
            config=make_generation_config(tools=[trip_planning_tool_schema], temperature=0.0)
        )
        total_tokens += llm_response.usage_metadata.total_token_count

        all_function_calls = [
            part.function_call
            for part in llm_response.candidates[0].content.parts
            if hasattr(part, 'function_call') and part.function_call
        ]

        if not all_function_calls:
            final_answer = extract_text_from_response(llm_response)
            print(f'\n[Step {step_number}] FINAL ANSWER:')
            print(final_answer)
            print(f'\n📊 Steps: {step_number} | Tokens: {total_tokens}')
            return final_answer

        tool_results = []
        for function_call in all_function_calls:
            result = TRIP_TOOL_MAP[function_call.name](**dict(function_call.args))
            tool_results.append(result)
            print(f'[Step {step_number}] ACTION  → {function_call.name}({dict(function_call.args)})')
            print(f'[Step {step_number}] RESULT  → {json.dumps(result, indent=2)[:200]}...')
            print()

        conversation.append(llm_response.candidates[0].content)  # preserves thought_signature
        conversation.append(types.Content(role='user', parts=[
            types.Part(function_response=types.FunctionResponse(
                name=function_call.name,
                response={'result': result}
            ))
            for function_call, result in zip(all_function_calls, tool_results)
        ]))

    return 'Max steps reached.'

run_trip_planning_agent(
    "I want to fly from Mumbai to Berlin on 2025-08-15 and stay for 3 nights. "
    "Find me the cheapest flight and a hotel with at least 4-star rating. "
    "What's the total cost in INR?"
)

TASK: I want to fly from Mumbai to Berlin on 2025-08-15 and stay for 3 nights. Find me the cheapest flight and a hotel with at least 4-star rating. What's the total cost in INR?
[Step 1] ACTION  → search_flights({'origin': 'Mumbai', 'destination': 'Berlin', 'date': '2025-08-15'})
[Step 1] RESULT  → {
  "route": "Mumbai\u2192Berlin",
  "date": "2025-08-15",
  "options": [
    {
      "flight_id": "SK201",
      "airline": "SkyWings",
      "price_usd": 580,
      "duration_h": 9
    },
    {
    ...

[Step 2] ACTION  → search_hotels({'checkin': '2025-08-15', 'checkout': '2025-08-18', 'city': 'Berlin'})
[Step 2] RESULT  → {
  "city": "Berlin",
  "checkin": "2025-08-15",
  "checkout": "2025-08-18",
  "options": [
    {
      "hotel_id": "H01",
      "name": "CityInn",
      "price_per_night_usd": 85,
      "rating": 3.8...

[Step 3] ACTION  → calculate_total_trip_cost({'flight_price_usd': 520, 'hotel_price_per_night_usd': 140, 'nights': 3})
[Step 3] RESULT  → {
  "flight_usd": 520,
  "hot

'For your trip from Mumbai to Berlin (2025-08-15 to 2025-08-18), here are the best options found:\n\n*   **Cheapest Flight:** SwiftAir (SW900) at **$520**.\n*   **4-Star Hotel:** ParkView (Rating: 4.3) at **$140 per night**.\n\nThe total cost for the 3-night stay and flight is **78,490 INR** (approximately $940 USD).'

---
## Demo 3: Agent Self-Correction — When Tools Fail

Real tools fail. A robust agent should recognize an error and try an alternative strategy.

In [22]:
# Track how many times the flaky tool has been called so we can make it fail only once
flight_search_call_count = {'total': 0}

def search_flights_flaky(origin: str, destination: str, date: str) -> dict:
    """Simulates a real-world API that fails on the first call.

    On attempt 1: raises a ValueError with a hint (try a nearby airport).
    On attempt 2+: returns normally.
    The agent receives the error as an observation and reasons about what to do next.
    """
    flight_search_call_count['total'] += 1
    if flight_search_call_count['total'] == 1:
        raise ValueError(
            'API_ERROR: Flight search service temporarily unavailable. Try a nearby airport.'
        )
    return {
        'route':   f'{origin}→{destination}',
        'date':    date,
        'options': [
            {'flight_id': 'BK99', 'airline': 'BackupAir', 'price_usd': 610, 'duration_h': 10}
        ]
    }

def run_agent_with_error_handling(task: str, max_steps: int = 10):
    """Same ReAct loop as before, but tool errors are caught and fed back to the LLM.

    Instead of crashing on a tool exception, we return {'error': message} as the
    tool result. The LLM sees the error as an observation and can reason about it:
    retry with different args, use an alternative tool, or apologise and stop.
    """
    conversation = [
        types.Content(role='user', parts=[types.Part(text=
            'You are a helpful travel agent. If a tool returns an error, '
            'reason about it and try an alternative approach. ' + task
        )])
    ]
    print('=' * 65)
    print(f'TASK: {task}')
    print('=' * 65)

    for step_number in range(1, max_steps + 1):
        llm_response = gemini_client.models.generate_content(
            model=GEMINI_MODEL,
            contents=conversation,
            config=make_generation_config(tools=[trip_planning_tool_schema], temperature=0.2)
        )
        all_function_calls = [
            part.function_call
            for part in llm_response.candidates[0].content.parts
            if hasattr(part, 'function_call') and part.function_call
        ]

        if not all_function_calls:
            print(f'\n[Step {step_number}] FINAL ANSWER:\n{extract_text_from_response(llm_response)}')
            return

        # Execute tools; on exception, return the error string as the observation
        # so the LLM can reason about what went wrong
        tool_responses = []
        for function_call in all_function_calls:
            print(f'[Step {step_number}] ACTION → {function_call.name}({dict(function_call.args)})')
            try:
                if function_call.name == 'search_flights':
                    tool_result = search_flights_flaky(**dict(function_call.args))
                else:
                    tool_result = TRIP_TOOL_MAP[function_call.name](**dict(function_call.args))
                print(f'[Step {step_number}] OK     → {tool_result}')
                tool_responses.append({'name': function_call.name, 'response': {'result': tool_result}})
            except Exception as tool_error:
                error_text = str(tool_error)
                print(f'[Step {step_number}] ERROR  → {error_text}')
                print(f'           Agent will see this error and decide what to do next...')
                tool_responses.append({'name': function_call.name, 'response': {'error': error_text}})

        conversation.append(llm_response.candidates[0].content)  # preserves thought_signature
        conversation.append(types.Content(role='user', parts=[
            types.Part(function_response=types.FunctionResponse(
                name=tool_resp['name'],
                response=tool_resp['response']
            ))
            for tool_resp in tool_responses
        ]))
        print()

run_agent_with_error_handling(
    "Find a flight from Mumbai to Berlin on 2025-08-20. Stay for 2 nights."
)

TASK: Find a flight from Mumbai to Berlin on 2025-08-20. Stay for 2 nights.
[Step 1] ACTION → search_flights({'date': '2025-08-20', 'origin': 'Mumbai', 'destination': 'Berlin'})
[Step 1] ERROR  → API_ERROR: Flight search service temporarily unavailable. Try a nearby airport.
           Agent will see this error and decide what to do next...

[Step 2] ACTION → search_flights({'date': '2025-08-20', 'origin': 'Delhi', 'destination': 'Berlin'})
[Step 2] OK     → {'route': 'Delhi→Berlin', 'date': '2025-08-20', 'options': [{'flight_id': 'BK99', 'airline': 'BackupAir', 'price_usd': 610, 'duration_h': 10}]}

[Step 3] ACTION → search_hotels({'checkin': '2025-08-20', 'checkout': '2025-08-22', 'city': 'Berlin'})
[Step 3] OK     → {'city': 'Berlin', 'checkin': '2025-08-20', 'checkout': '2025-08-22', 'options': [{'hotel_id': 'H01', 'name': 'CityInn', 'price_per_night_usd': 85, 'rating': 3.8}, {'hotel_id': 'H02', 'name': 'ParkView', 'price_per_night_usd': 140, 'rating': 4.3}, {'hotel_id': 'H03', 'na

---
## Demo 4: Chain vs. Agent — When to Use Which

An agent is powerful but expensive. Sometimes a fixed **prompt chain** is the right tool:  
faster, cheaper, fully deterministic. Run the same task both ways and compare.

In [23]:
COMPARISON_TASK = 'Find a flight from Delhi to Paris on 2025-09-10 and a hotel for 4 nights. Summarize the cheapest options.'

# ── Approach A: Fixed 3-step prompt chain ─────────────────────────────────
# Steps are hardcoded: search flights → search hotels → ask LLM to summarize.
# No agent loop. The LLM only writes the final sentence; it never decides what to do.

chain_start_time = time.time()

flight_options = search_flights('Delhi', 'Paris', '2025-09-10')
hotel_options  = search_hotels('Paris', '2025-09-10', '2025-09-14')

chain_llm_response = gemini_client.models.generate_content(
    model=GEMINI_MODEL,
    contents=(
        f'Summarize the cheapest flight and hotel options from this data:\n'
        f'Flights: {flight_options}\nHotels: {hotel_options}'
    ),
    config=make_generation_config(temperature=0.0)
)

chain_elapsed_seconds = round(time.time() - chain_start_time, 2)
chain_token_count     = chain_llm_response.usage_metadata.total_token_count

print('APPROACH A — Fixed Chain')
print(extract_text_from_response(chain_llm_response))
print(f'\n⏱  Time: {chain_elapsed_seconds}s | 🪙 Tokens: {chain_token_count}')

print('\n' + '─' * 65 + '\n')

# ── Approach B: ReAct agent ────────────────────────────────────────────────
# The agent receives the task and decides on its own which tools to call.
# It can handle variations (e.g. ambiguous dates, extra constraints) that the
# hardcoded chain would silently mishandle.

agent_start_time = time.time()
run_trip_planning_agent(COMPARISON_TASK)
agent_elapsed_seconds = round(time.time() - agent_start_time, 2)

print(f'⏱  Total agent time: {agent_elapsed_seconds}s')
print('\n' + '=' * 65)
print('SUMMARY')
print('=' * 65)
print(f'Chain  : {chain_elapsed_seconds}s, {chain_token_count} tokens — fast, deterministic, inflexible')
print(f'Agent  : {agent_elapsed_seconds}s — flexible, handles ambiguity, more expensive')
print('\nRule of thumb: use a chain when the steps are known in advance;')
print('use an agent when the steps depend on intermediate results or user intent.')

APPROACH A — Fixed Chain
Here is the summary of the cheapest travel options for your trip to Paris (Sept 10–14, 2025):

*   **Cheapest Flight:** SwiftAir (Flight SW900) at **$520**.
*   **Cheapest Hotel:** BudgetStay at **$55 per night** ($220 total for 4 nights).

**Total estimated cost for the cheapest options: $740.**

⏱  Time: 0.92s | 🪙 Tokens: 408

─────────────────────────────────────────────────────────────────

TASK: Find a flight from Delhi to Paris on 2025-09-10 and a hotel for 4 nights. Summarize the cheapest options.
[Step 1] ACTION  → search_flights({'destination': 'Paris', 'origin': 'Delhi', 'date': '2025-09-10'})
[Step 1] RESULT  → {
  "route": "Delhi\u2192Paris",
  "date": "2025-09-10",
  "options": [
    {
      "flight_id": "SK201",
      "airline": "SkyWings",
      "price_usd": 580,
      "duration_h": 9
    },
    {
      ...

[Step 1] ACTION  → search_hotels({'checkout': '2025-09-14', 'city': 'Paris', 'checkin': '2025-09-10'})
[Step 1] RESULT  → {
  "city": "Paris

In [24]:
# ✏️  Give the agent a multi-step task of your own.
#    The agent has access to: get_weather, convert_currency, get_flight_info
#    Try to craft a question that requires at least 2 tool calls.

my_task = "TODO: write a multi-step travel question here"
run_react_agent(my_task)

TASK: TODO: write a multi-step travel question here

[Step 1] THOUGHT → Final answer (no more tools needed)
[Step 1] ANSWER  → I'm ready to help! Please go ahead and provide your multi-step travel question. 

For example, you could ask something like:
*   "What is the weather like in Tokyo right now, and how much would 500 USD be in Japanese Yen?"
*   "Are there any flights from New York to London, and what is the current weather in London?"

Just let me know what you need!

📊 Total steps: 1 | Total tokens used: 309


'I\'m ready to help! Please go ahead and provide your multi-step travel question. \n\nFor example, you could ask something like:\n*   "What is the weather like in Tokyo right now, and how much would 500 USD be in Japanese Yen?"\n*   "Are there any flights from New York to London, and what is the current weather in London?"\n\nJust let me know what you need!'

---
## Key Takeaways

| Concept | What it means |
|---|---|
| ReAct loop | Agent alternates Thought (reason) → Action (tool) → Observation until done |
| Agent = while-loop | No framework magic — just a loop that stops when no tool is called |
| Self-correction | Pass tool errors back to the LLM; it reasons about failures and retries |
| max_steps guard | Always cap the loop — prevents runaway agents burning tokens |
| Token growth | Each step adds to the context; long agents get expensive fast |
| Chain vs Agent | Fixed chain = faster/cheaper; agent = flexible but costly. Match to task. |